# 01 — Features, labels, and alignment

This notebook converts raw OHLCV snapshots into feature/label rows.

The label is:
```text
1 if stock future return over horizon > Nifty future return over horizon else 0
```

The key alignment fields are:

```text
stock_id + date
```

In [ ]:
# Colab setup: make sure we are inside the cloned repo.
import os
REPO_DIR = "/content/nifty50-multimodal-transformer"
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
from pathlib import Path
import pandas as pd

from src.data.download_yfinance import deterministic_csv_path_for_ticker
from src.data.features import compute_technical_features
from src.data.labels import generate_outperformance_label

TICKERS = ["RELIANCE.NS", "TCS.NS", "INFY.NS"]
BENCHMARK = "^NSEI"
OUTPUT_DIR = Path("data/processed/real_world_demo")
RAW_DIR = OUTPUT_DIR / "raw"

HORIZON_DAYS = 3

FEATURE_COLUMNS = [
    "log_return_1d",
    "cum_return_3d",
    "cum_return_5d",
    "cum_return_10d",
    "realized_vol_5d",
    "realized_vol_10d",
    "high_low_range_over_close",
    "close_over_10dma_minus_1",
    "close_over_20dma_minus_1",
    "volume_over_20d_avg",
    "stock_minus_index_return",
]

In [ ]:
benchmark_df = pd.read_csv(deterministic_csv_path_for_ticker(BENCHMARK, RAW_DIR), parse_dates=["date"])

frames = []
for ticker in TICKERS:
    stock_df = pd.read_csv(deterministic_csv_path_for_ticker(ticker, RAW_DIR), parse_dates=["date"])
    features = compute_technical_features(stock_df, benchmark_df)
    labelled = generate_outperformance_label(features, horizon_days=HORIZON_DAYS)
    labelled["stock_id"] = ticker
    frames.append(labelled)

combined = pd.concat(frames, ignore_index=True)
required = ["stock_id", "date", "label", *FEATURE_COLUMNS]
tabular_df = combined.dropna(subset=required).loc[:, required + ["close", "volume"]].reset_index(drop=True)

tabular_csv = OUTPUT_DIR / "tabular_samples.csv"
tabular_df.to_csv(tabular_csv, index=False)
print("Wrote:", tabular_csv)
print("Rows:", len(tabular_df))
display(tabular_df.head())

In [ ]:
# Label distribution by stock.
label_summary = tabular_df.groupby("stock_id")["label"].agg(["count", "sum", "mean"]).rename(columns={"sum": "positive_labels", "mean": "positive_rate"})
display(label_summary)

In [ ]:
# Show the sample timeline per stock.
timeline = tabular_df.groupby("stock_id")["date"].agg(["min", "max", "count"])
display(timeline)

## Checkpoint output

This notebook writes:

```text
data/processed/real_world_demo/tabular_samples.csv
```

Next notebook: `02_build_multimodal_artifact.ipynb`.